# Create dirichlet distributions for base and high skew

In [50]:
# Configuration
data_path = '../data'
num_participants = 8
quantity_alpha = 1.5 # 1.2
label_alpha = 0.8 # 0.6

In [51]:
# Imports
import numpy as np
import keras
import pandas as pd
import sklearn
import os
from sklearn.preprocessing import LabelEncoder

# Helper function for CIFAR100
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict

In [52]:
# Dataset definition
class CIFAR10Dataset():
    num_labels = 10
    unique_labels = [0,1,2,3,4,5,6,7,8,9]
    num_samples = 60000
    num_features = 3072
    num_samples_per_label = [6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]
    train_test_separated = False

    def __init__(self):
        (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test
    
    def load(self) -> np.ndarray:
        assert self.x_train.shape == (50000, 32, 32, 3)
        assert self.x_test.shape == (10000, 32, 32, 3)
        assert self.y_train.shape == (50000, 1)
        assert self.y_test.shape == (10000, 1)
        
        data_x = np.concatenate((self.x_train, self.x_test))
        data_y = np.concatenate((self.y_train, self.y_test))
        
        data = np.array(list(zip(data_x, data_y)), dtype=object)
        
        assert data.shape == (CIFAR10Dataset.num_samples,2)
        return data
    
class CIFAR100Dataset():

    num_labels = 100
    unique_labels = [x for x in range(0,100)]
    num_samples = 60000
    num_features = 3072
    data_path = f"{data_path}/cifar100"
    num_samples_per_label = [600 for x in range(0,100)]
    train_test_separated = False

    def __init__(self):
        train_data = unpickle(f"{CIFAR100Dataset.data_path}/train")
        self.x_train = train_data[b'data']
        self.y_train = np.array(train_data[b'fine_labels'])

        test_data = unpickle(f"{CIFAR100Dataset.data_path}/test")
        self.x_test = test_data[b'data']
        self.y_test = np.array(test_data[b'fine_labels'])
    
    def load(self) -> np.ndarray:

        data_x = np.concatenate((self.x_train, self.x_test))
        data_y = np.concatenate((self.y_train, self.y_test))
        
        data_x = data_x.reshape(len(data_x),3,32,32).transpose(0,2,3,1)
        
        assert data_x.shape == (60000, 32, 32, 3)
        assert data_y.shape == (60000,)
        
        data = np.array(list(zip(data_x, data_y)), dtype=object)
        
        assert data.shape == (CIFAR100Dataset.num_samples,2)
        return data

class AdultDataset():

    num_labels = 2
    unique_labels = [0,1]
    num_samples = 48841
    num_features = 14
    data_path = f"{data_path}/adult"
    num_samples_per_label = [37154, 11687]
    train_test_separated = False
    
    def __init__(self):
        column_names = [
            "age", "workclass", "fnlwgt", "education", "education-num",
            "marital-status", "occupation", "relationship", "race", "sex",
            "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
        ]
 
        train_data = pd.read_csv(f'{AdultDataset.data_path}/adult.data', names=column_names, na_values=' ?', skipinitialspace=True)
        
        test_data = pd.read_csv(f'{AdultDataset.data_path}/adult.test', names=column_names, na_values=' ?', skipinitialspace=True, skiprows=1)
        
        data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
        data = data.dropna()

        categorical_columns = [
            'workclass', 'education', 'marital-status', 'occupation', 'relationship',
            'race', 'sex', 'native-country'
        ]
        numerical_columns = [
            'age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week'
        ]
        
        target_column = 'income'
        data[target_column] = data[target_column].apply(lambda x: 1 if x.strip().startswith('>') else 0)
  
        for col in categorical_columns:
            data[col] = LabelEncoder().fit_transform(data[col])
            
        self.data_x = data[categorical_columns + numerical_columns]
        self.data_y = data[target_column]

    def load(self) -> np.ndarray:
        data = np.array(list(zip(self.data_x.values, self.data_y.values)), dtype=object)
        return data
  
class CoverTypeDataset():
    
    num_labels = 7
    unique_labels = [0,1,2,3,4,5,6]
    num_samples = 581012
    num_features = 54
    num_samples_per_label = [211840, 283301,  35754,   2747,   9493,  17367,  20510]
    data_path = f"{data_path}/covertype"
    train_test_separated = False
    
    def __init__(self):
        column_names = [
            "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
            "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
            "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
            "Horizontal_Distance_To_Fire_Points"
        ] + [f"Wilderness_Area_{i}" for i in range(4)] + \
        [f"Soil_Type_{i}" for i in range(40)] + ["Cover_Type"]

        data = pd.read_csv(f"{CoverTypeDataset.data_path}/covtype.data.gz", header=None, names=column_names)

        wilderness_cols = [f'Wilderness_Area_{i}' for i in range(4)]
        soil_cols = [f'Soil_Type_{i}' for i in range(40)]

        data['Wilderness_Area'] = data[wilderness_cols].idxmax(axis=1).str.split('_').str[-1].astype(int)
        data['Soil_Type'] = data[soil_cols].idxmax(axis=1).str.split('_').str[-1].astype(int)

        target_column = 'Cover_Type'

        data = data.drop(columns=wilderness_cols + soil_cols)

        data[target_column] = data[target_column] - 1

        self.x = data.drop(columns=target_column)
        self.y = data[target_column]


    def load(self) -> np.ndarray:
        data = np.array(list(zip(self.x.values, self.y.values)), dtype=object)
        return data
  

### 1. Calculate the partitions 

In [53]:
def label_and_quantity_parition(total_num_labels, total_num_samples, total_num_samples_per_label, num_participants: int, quantity_alpha: float, label_alpha: float, measure_valuable_impact: bool) -> list:
    
    """ Data Modification - Quantity and Label Skew - for balanaced and unbalanced datasets
    Summary:
        # 1. Start with the label skew and then adjust to the quantity skew
        # 2. Create a delta to balance the behviour
        # 3. Redistribute the samples to match the quantity skew
    Limitations:
        # 1. May not work well with extreme skews and small datasets
        # 2. High unbalancedness may cause the method to not find a suitable selection
    """
    
    # 1. Set the seed for reproducibility
    np.random.seed(24)

    # 2. Define necessary information about the dataset
    num_labels = total_num_labels
    num_samples = total_num_samples
    num_samples_per_label = np.array(total_num_samples_per_label)

    # brief validation
    assert sum(num_samples_per_label) == num_samples

    # 3. Generate label distribution for each client
    # This determines what fraction of each label's total samples goes to each client.
    label_proportions_per_participant = np.random.dirichlet([label_alpha] * num_participants, size=num_labels).T

    # 4. Generate quantity distribution for each client
    quantity_proportions_per_participant = np.random.dirichlet([quantity_alpha] * num_participants, size=1).flatten()

    # 5. Calculate the target number of samples for each client based on quantity skew
    target_samples_per_client = (quantity_proportions_per_participant * num_samples).astype(int)
    
    # 6. Adjust for rounding errors to ensure total samples are conserved
    sample_diff = num_samples - target_samples_per_client.sum()
    target_samples_per_client[np.argmax(target_samples_per_client)] += sample_diff
    assert target_samples_per_client.sum() == num_samples

    samples_per_label_per_participant = (label_proportions_per_participant * num_samples_per_label).astype(int)

    # 7. Adjust allocations to meet the target sample counts per client (quantity skew)
    current_samples_per_client = samples_per_label_per_participant.sum(axis=1)
    delta = current_samples_per_client - target_samples_per_client

    for i in range(num_participants):
        if delta[i] > 0: 
            to_give = delta[i]
            # Give away samples proportionally to their current label distribution
            proportions = samples_per_label_per_participant[i] / samples_per_label_per_participant[i].sum()
            give_away = (proportions * to_give).astype(int)
            
            # Adjust for rounding
            give_away_diff = to_give - give_away.sum()
            give_away[np.argmax(give_away)] += give_away_diff
            
            samples_per_label_per_participant[i] -= give_away
            delta[i] -= give_away.sum()
    
    # 7.1 Calculate total samples to be redistributed for each label
    total_to_redistribute = num_samples_per_label - samples_per_label_per_participant.sum(axis=0)

    # 7.2 Identify clients that need more samples
    clients_in_need_indices = np.where(delta < 0)[0]

    # 7.3 Distribute remaining samples to clients that are under their target
    if len(clients_in_need_indices) > 0:
        
        for label_idx in range(num_labels):
            samples_to_distribute = total_to_redistribute[label_idx]
            if samples_to_distribute == 0:
                continue

            # Proportions for clients in need for the current label
            need_proportions = np.array([label_proportions_per_participant[i, label_idx] for i in clients_in_need_indices])
            if need_proportions.sum() == 0:
                need_proportions = np.ones(len(clients_in_need_indices)) / len(clients_in_need_indices)
            else:
                need_proportions /= need_proportions.sum()

            
            alloc = (need_proportions * samples_to_distribute).astype(int)
            
            # Adjust for rounding
            alloc_diff = samples_to_distribute - alloc.sum()
            alloc[np.argmax(alloc)] += alloc_diff
            
            for i, client_idx in enumerate(clients_in_need_indices):
                samples_per_label_per_participant[client_idx, label_idx] += alloc[i]

    # 7.4 Vaildate correct dististribution
    assert samples_per_label_per_participant.sum() == num_samples
    assert np.all(samples_per_label_per_participant.sum(axis=0) == num_samples_per_label)

    # 7.5 For simulation: Move the highest value participant to the first spot in the list
    if measure_valuable_impact:
        first_row = samples_per_label_per_participant[0].copy()
        max_row_index = np.argmax(np.array(samples_per_label_per_participant).sum(axis=1))
        samples_per_label_per_participant[0] = samples_per_label_per_participant[max_row_index]
        samples_per_label_per_participant[max_row_index] = first_row
    
    # 7.6 Remove all single label assignments to avoid clients with only one sample of a label
    # This is necessary to prevent the stratified train test split from breaking during participant setup
    samples_per_label_per_participant[samples_per_label_per_participant == 1] = 0
    
    return samples_per_label_per_participant.tolist()

### 2. Distribute the dataset samples across the partitions

In [54]:
def partition_data(dataset,
                    num_participants:int,
                    total_num_labels:int,
                    num_samples_per_label_per_client):
    
    """The dataset for client i is created by iterating over the list of num samples per label for the client and taking the specified number
        of samples per label from the dataset.
        Here, the dataset is modified into multiple collections (one per label), from which the required number of samples is then taken.
        A collection is simply a sequence in which each participant starts at a different point: one participant starts at the beginning, the next at the end of the first participant, and so on.
    """
    
    # 1. Create the pods
    grouped_datasets = []
    data = dataset.load()
    df = pd.DataFrame(data)

    for label in dataset.unique_labels:
        grouped_datasets.append(df[df[1]==label].to_numpy())

    taken_data_per_label = [0 for x in range(0,len(dataset.unique_labels))]
    
    # 2. Iterate over the clients and create the datasets
    client_datasets = {}
    for i in range(num_participants):
        
        client_samples_per_label = num_samples_per_label_per_client[i]
        client_dataset = np.ndarray(shape=(0,2))

        for j in range(total_num_labels):
            
            client_dataset = np.concatenate((client_dataset,grouped_datasets[j][taken_data_per_label[j]:(taken_data_per_label[j]+client_samples_per_label[j])]))
            taken_data_per_label[j] = taken_data_per_label[j] + client_samples_per_label[j]

        assert len(client_dataset) == sum(client_samples_per_label)
        client_datasets[f"{i}"] = client_dataset

    return client_datasets

### 3. Store the generated partitions

In [55]:
from numpy.random import dirichlet


def split_data(num_participants, dataset, client_datasets, train_test_split=0.3):
    
    client_split_data = {}
    for i in range(num_participants):
                
        if dataset.train_test_separated:
            x_train = np.array([x for x, y in client_datasets[f"{i}"][0]])
            y_train = np.array([y for x, y in client_datasets[f"{i}"][0]])
            
            x_test = np.array([x for x, y in client_datasets[f"{i}"][1]])
            y_test = np.array([y for x, y in client_datasets[f"{i}"][1]])
            
        else:
            data_x, data_y = zip(*client_datasets[f"{i}"])
            istime_series = False
            if hasattr(dataset, 'timeseries'):
                istime_series = dataset.timeseries
            
            if istime_series:
                split_idx = int(len(data_x) * float(1-train_test_split))
                x_train, x_test = data_x[:split_idx], data_x[split_idx:]
                y_train, y_test = data_y[:split_idx], data_y[split_idx:]
            else:
                data_x = np.array(data_x)
                data_y = np.array(data_y)
                x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(data_x, data_y, stratify=data_y, test_size=0.3, random_state=24)
        
        client_split_data[str(i)] = (np.array(list(zip(x_train, y_train)),dtype=object),
                np.array(list(zip(x_test,y_test)),dtype=object))
    return client_split_data

def store_data(num_participants,file_path, client_datasets_dict):
    
        file_path = f"{data_path}/dirichlet_partitions"

        for i in range(num_participants):
            if not os.path.exists(f"{file_path}/client-{i}"):
                os.makedirs(f"{file_path}/client-{i}")
            
            np.save(f"{file_path}/client-{i}/client-data.npy", client_datasets_dict[f"{i}"][0])
            np.save(f"{file_path}/client-{i}/client-data-test.npy", client_datasets_dict[f"{i}"][1])

        return True

## Create a partition for a dataset (Example: CIFAR10)

In [56]:
# Select dataset
dataset = CIFAR10Dataset()

# Calculate partitions
num_samples_per_label_per_client = label_and_quantity_parition(dataset.num_labels, dataset.num_samples, dataset.num_samples_per_label, num_participants=num_participants, quantity_alpha=quantity_alpha, label_alpha=label_alpha, measure_valuable_impact=True)

client_dataset = partition_data(dataset=dataset, num_participants=num_participants, total_num_labels=dataset.num_labels, num_samples_per_label_per_client=num_samples_per_label_per_client)

client_split_data = split_data(num_participants=num_participants, dataset=dataset, client_datasets=client_dataset)

store_data(num_participants=num_participants,file_path=data_path, client_datasets_dict=client_split_data)

True